# Project 15: Text2SQL via Prompt Engineering

# Submission Details

Can you create dataset for employees, and share the results in a notebook?

#Retrive Data

In [15]:
!curl "https://api.mockaroo.com/api/2f5fff80?count=1000&key=64edf120" > "Employee.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  105k    0  105k    0     0  56513      0 --:--:--  0:00:01 --:--:-- 56508


#Database setup

In [16]:
import sqlite3 #python sql db
import pandas as pd
import os

In [17]:
employee_schema="""
CREATE TABLE IF NOT EXISTS employee(
employee_id INT PRIMARY KEY,
first_name VARCHAR(50),
last_name VARCHAR(50),
gender VARCHAR(50),
date_of_birth DATE,
job_title VARCHAR(50),
department VARCHAR(50),
hire_date DATE,
email VARCHAR(50),
salary INT);
"""

In [18]:
db_name="employee_dataset.db"
if os.path.exists(db_name):
  os.remove(db_name)
  print(f"Removed existing database '{db_name}'.")

Removed existing database 'employee_dataset.db'.


In [20]:
COLUMN_DATA_TYPES={
    'employee':{
        'employee_id': 'int64',
        'first_name' : 'object',
        'last_name' : 'object',
        'gender' : 'object',
        'date_of_birth' : 'datetime64[ns]',
        'job_title' : 'object',
        'department' : 'object',
        'hire_date' : 'datetime64[ns]',
        'email' : 'object',
        'salary' : 'int64'
    }
}

In [21]:
conn=None

try:
  conn=sqlite3.connect(db_name)
  cursor=conn.cursor()
  print(f"DB: {db_name} successfully created and connected")

  cursor.execute(employee_schema)
  cursor.execute(salary_schema)

  print("Tables: employee and salary created successfully")

  csv_to_schema_map={
      '/content/Employee.csv' : 'employee'
  }

  for csv,schema in csv_to_schema_map.items():
    if os.path.exists(csv):
      print(f"\nProcessing {csv} for table {schema} ...")

      df=pd.read_csv(csv)
      expected_schema=COLUMN_DATA_TYPES[schema]
      expected_cols=list(expected_schema.keys())

      # Drop the manager_id column from the salary DataFrame if it exists
      if schema == 'salary' and 'manager_id' in df.columns:
          df = df.drop(columns=['manager_id'])


      df=df[df.columns.intersection(expected_cols)]

      for col in expected_cols:
        if col not in df.columns:
          df[col]=None

      df=df[expected_cols]

      for col,dtype in expected_schema.items():
        if 'datetime64[ns]' in dtype:
          df[col]=pd.to_datetime(df[col],errors='coerce')
        else:
          try:
            df[col]=df[col].astype(dtype)
          except (ValueError, TypeError) as e:
            print(f" -WARNING: could not convert {col} to {dtype}. Error {e}")
      df.to_sql(schema,conn,if_exists='append',index=False)
      print(f" ->Data from {csv} loaded into {schema} succefully :) ")
    else:
      print(f"WARNING: {csv} not found. Skipping data load for {schema}")
  conn.commit()
  print("Data committed successfully !!")
except sqlite3.Error as e:
  print(f"Database error: {e}")
except pd.errors.EmptyDataError as e:
  print(f"Pandas error: {e}. One of the CSV files might be empty")
except KeyError as e:
  print(f"Schema definition error: A column is missing from TABLE_DATA_TYPE dictionary: {e} ")
except Exception as e:
  print(f"An unexpected error: {e}")
finally:
  if conn:
    conn.close()
    print("DB connection closed")

DB: employee_dataset.db successfully created and connected
Tables: employee and salary created successfully

Processing /content/Employee.csv for table employee ...
 ->Data from /content/Employee.csv loaded into employee succefully :) 
Data committed successfully !!
DB connection closed


#install genAI libraries

In [22]:
!pip install google-genai

In [23]:
from google import genai
from google.colab import userdata

In [25]:
genai_client=genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

#PROMPT ENGINEERING

In [26]:
prompt = """
### **ROLE**

You are an expert-level SQLite Database Engineer specializing in Natural Language to SQL (NL2SQL) translation. Your sole function is to convert user questions written in plain English into accurate, efficient, and syntactically correct SQLite queries based on a fixed database schema.

-----

### **CONTEXT**

You are the core translation engine for a business intelligence dashboard. This tool allows non-technical employees to query the company's employee database using natural language. The database dialect is always **SQLite**. Your responses will be executed directly on the database.

The database consists of the following table:

**`employee` table:**

```sql
CREATE TABLE employee(
employee_id INT PRIMARY KEY,
first_name VARCHAR(50),
last_name VARCHAR(50),
gender VARCHAR(50),
date_of_birth DATE,
job_title VARCHAR(50),
department VARCHAR(50),
hire_date DATE,
email VARCHAR(50),
salary INT);
```

-----

### **TASK**

Your task is to receive a user's question in natural language and convert it into a single, executable SQLite query. Follow these steps meticulously:

1.  **Analyze the User's Query:** Deconstruct the user's question to understand their core intent. Identify the specific data, conditions, aggregations (like `SUM`, `COUNT`, `AVG`), and ordering they are asking for.
2.  **Map to the Schema:** Map the entities from the user's query to the appropriate table (`employee`) and columns.
3.  **Construct the SQLite Query:** Write a clean and efficient `SELECT` statement that is syntactically correct for SQLite. Ensure all table and column names are accurate.
4.  **Handle Ambiguity:** If the user's query is vague, ambiguous, or lacks the necessary information to create a precise query, do not guess. Instead, formulate a specific, targeted question to ask the user for the missing information.

-----

### **CONSTRAINTS**

  * **Read-Only Operations:** You must **ONLY** generate `SELECT` queries. Never generate `INSERT`, `UPDATE`, `DELETE`, `DROP`, or any other data-modifying statements.
  * **Adhere Strictly to Schema:** Only use the tables and columns defined in the context. Do not invent or assume the existence of any other tables or columns.
  * **No Explanations:** Do not add any conversational text or explanations about the query you generate. Your output must strictly follow the specified format.
  * **Single Query Only:** The final output must be a single, complete, and executable SQL query.
  * **Handle Impossibility:** If a request is impossible to fulfill with the given schema (e.g., "Which employee made the most sales?"), state clearly that the request cannot be completed and briefly explain why.

-----

### **EXAMPLES**

**Example 1: Simple Lookup**

  * **User Query:** "Show me all employees in the Training department"
  * **Expected Output:**
    ```json
    {
      "status": "success",
      "response": "SELECT * FROM employee WHERE department = 'Training';"
    }
    ```

**Example 2: Complex Aggregation**

  * **User Query:** "What is the average salary of an employee in Engineering department"
  * **Expected Output:**
    ```json
    {
      "status": "success",
      "response": "SELECT Engineering,AVG(SALARY) AS AVERAGE_SALARY FROM EMPLOYEE;"
    }
    ```

**Example 3: Ambiguous Query**

  * **User Query:** "Show me the head of everyone"
  * **Expected Output:**
    ```json
    {
      "status": "clarification_needed",
      "response": "Could you please define what 'head' means? For example, 'of HR department'."
    }
    ```

**Example 4: Impossible Query**

  * **User Query:** "Which employee should be promoted"
  * **Expected Output:**
    ```json
    {
      "status": "error",
      "response": "I cannot answer this question as the database does not contain information about employee performance."
    }
    ```

-----

### **OUTPUT FORMAT**

Your final response must be a single JSON object with two keys:

1.  `"status"`: A string with one of three possible values: `"success"`, `"clarification_needed"`, or `"error"`.
2.  `"response"`:
      * If `status` is `"success"`, this will be a string containing the complete SQLite query.
      * If `status` is `"clarification_needed"`, this will be a string containing the clarifying question for the user.
      * If `status` is `"error"`, this will be a string explaining why the query could not be generated.
"""

In [27]:
import json

def get_sql_query(genai_client,prompt,user_q):
  contents=f"""
{prompt}
  Here's the user query in english you need to work on:
  {user_q}
   """

  response=genai_client.models.generate_content(model='gemini-2.5-flash',contents=contents)
  usage_data=response.usage_metadata
  print(f"Input token count: {usage_data.prompt_token_count}")
  print(f"Thought token count: {usage_data.thoughts_token_count}")
  print(f"Output token count: {usage_data.candidates_token_count}")
  print(f"Total token count: {usage_data.total_token_count}")

  output=json.loads(response.text.replace('```json','').replace('```',''))
  return output


In [28]:
def execute_query(query,db_name='employee_dataset.db'):
  conn=None
  try:
    conn=sqlite3.connect(db_name)
    cursor=conn.cursor()
    print(f"\nExecuting query on {db_name}: \n{query}")
    cursor.execute(query)
    result=cursor.fetchall()
    columns=[description[0] for description in cursor.description]
    results_as_dict=[dict(zip(columns,row)) for row in result]
    results_df=pd.DataFrame(results_as_dict)
    print("Query executed successfully")
    return results_df
  except sqlite3.Error as e:
    print(f"Database error while executing query: {e}")
    return None
  except Exception as e:
    print(f"An unexpected error occurred: {e}")
    return None
  finally:
    if conn:
      conn.close()


In [29]:
def text2sql(genai_client,prompt,user_q):
  op=get_sql_query(genai_client,prompt,user_q)
  if op['status']=='success':
    results=execute_query(op['response'])
    return results
  return op


In [31]:
text2sql(genai_client, prompt, "What is the hire date of employee with id 11")

Input token count: 1100
Thought token count: 282
Output token count: 37
Total token count: 1419

Executing query on employee_dataset.db: 
SELECT hire_date FROM employee WHERE employee_id = 11;
Query executed successfully


,hire_date
0,2012-06-06 00:00:00


In [32]:
text2sql(genai_client, prompt, "How many female employees are there")

Input token count: 1094
Thought token count: 53
Output token count: 43
Total token count: 1190

Executing query on employee_dataset.db: 
SELECT COUNT(employee_id) AS female_employee_count FROM employee WHERE gender = 'Female';
Query executed successfully


,female_employee_count
0,444


In [33]:
text2sql(genai_client, prompt, "Who is the highest paid in Engineering department")

Input token count: 1096
Thought token count: 193
Output token count: 48
Total token count: 1337

Executing query on employee_dataset.db: 
SELECT first_name, last_name, salary FROM employee WHERE department = 'Engineering' ORDER BY salary DESC LIMIT 1;
Query executed successfully


,first_name,last_name,salary
0,Kaitlynn,Warrender,9974133


In [34]:
text2sql(genai_client, prompt, "who has the highest experience and how much")

Input token count: 1096
Thought token count: 832
Output token count: 75
Total token count: 2003

Executing query on employee_dataset.db: 
SELECT first_name, last_name, (JULIANDAY('now') - JULIANDAY(hire_date)) / 365.25 AS years_of_experience FROM employee ORDER BY hire_date ASC LIMIT 1;
Query executed successfully


,first_name,last_name,years_of_experience
0,Shaun,Pereira,15.737945


In [35]:
text2sql(genai_client, prompt, "What is the number of employees in services")

Input token count: 1096
Thought token count: 41
Output token count: 37
Total token count: 1174

Executing query on employee_dataset.db: 
SELECT COUNT(employee_id) FROM employee WHERE department = 'Services';
Query executed successfully


,COUNT(employee_id)
0,80


In [36]:
text2sql(genai_client, prompt, "who is the expert in design engineering")

Input token count: 1095
Thought token count: 330
Output token count: 45
Total token count: 1470


{'status': 'error',
 'response': "I cannot determine who is an 'expert' as the database does not contain information about employee skill levels or expertise."}

In [37]:
text2sql(genai_client, prompt, "show in sorted order the number of male and females in each department")

Input token count: 1101
Thought token count: 727
Output token count: 77
Total token count: 1905

Executing query on employee_dataset.db: 
SELECT department, SUM(CASE WHEN gender = 'Male' THEN 1 ELSE 0 END) AS male_count, SUM(CASE WHEN gender = 'Female' THEN 1 ELSE 0 END) AS female_count FROM employee GROUP BY department ORDER BY department;
Query executed successfully


,department,male_count,female_count
0,Accounting,40,48
1,Business Development,34,29
2,Engineering,42,41
3,Human Resources,47,39
4,Legal,41,31
5,Marketing,44,42
6,Product Management,38,41
7,Research and Development,36,30
8,Sales,45,45
9,Services,32,39


In [38]:
text2sql(genai_client, prompt, "update the salary of employee with id 256")

Input token count: 1099
Thought token count: 82
Output token count: 61
Total token count: 1242


{'status': 'error',
 'response': 'I cannot fulfill this request. My function is limited to generating SELECT queries for retrieving data. I am not able to perform UPDATE, INSERT, DELETE, or any other data-modifying operations.'}

In [39]:
text2sql(genai_client, prompt, "give me the hierarchy of employees in the engineering department")

Input token count: 1098
Thought token count: 302
Output token count: 49
Total token count: 1449


{'status': 'error',
 'response': 'I cannot answer this question as the database schema does not contain information about employee hierarchy or reporting structures (e.g., manager IDs).'}

In [42]:
text2sql(genai_client, prompt, "give me the list of employees in the engineering department as per their experience and specify their experince in integer with no decimals")

Input token count: 1112
Thought token count: 784
Output token count: 89
Total token count: 1985

Executing query on employee_dataset.db: 
SELECT first_name, last_name, job_title, hire_date, CAST((julianday('now') - julianday(hire_date)) / 365.25 AS INT) AS years_of_experience FROM employee WHERE department = 'Engineering' ORDER BY years_of_experience DESC;
Query executed successfully


,first_name,last_name,job_title,hire_date,years_of_experience
0,Eimile,Stuckley,Account Coordinator,2010-05-31 00:00:00,15
1,Marijo,Raft,Assistant Manager,2010-03-02 00:00:00,15
2,Darrick,Bramsen,Compensation Analyst,2010-09-25 00:00:00,15
3,Tami,Corrin,Environmental Specialist,2010-03-09 00:00:00,15
4,Neille,Frizell,Pharmacist,2011-05-08 00:00:00,14
...,...,...,...,...,...
82,Anetta,Greveson,Geological Engineer,2021-06-12 00:00:00,4
83,Vikky,Keyse,Developer III,2021-05-02 00:00:00,4
84,Gunter,Rudd,Account Executive,2021-08-24 00:00:00,4
85,Demeter,Shorto,Developer II,2021-09-26 00:00:00,4


In [43]:
text2sql(genai_client, prompt, "visualise the top paid employees overall")

Input token count: 1095
Thought token count: 729
Output token count: 49
Total token count: 1873

Executing query on employee_dataset.db: 
SELECT employee_id, first_name, last_name, job_title, department, salary FROM employee ORDER BY salary DESC;
Query executed successfully


,employee_id,first_name,last_name,job_title,department,salary
0,899,Kaitlynn,Warrender,Speech Pathologist,Engineering,9974133
1,466,Marven,Tuberfield,Geological Engineer,Support,9963795
2,783,Jaclyn,Isgar,Electrical Engineer,Marketing,9949187
3,960,Davie,Sames,Director of Sales,Product Management,9928023
4,522,Aleta,Ramalho,Paralegal,Sales,9907078
...,...,...,...,...,...,...
995,220,Matthieu,Benedit,Technical Writer,Legal,61285
996,613,Brewster,Grelik,Environmental Specialist,Product Management,48278
997,343,Yves,Mathevon,Analyst Programmer,Sales,37231
998,579,Zena,Allawy,Software Test Engineer III,Business Development,19252


In [44]:
text2sql(genai_client, prompt, "visualise the top paid employees overall in a graphical structure")

Input token count: 1099
Thought token count: 298
Output token count: 71
Total token count: 1468


{'status': 'error',
 'response': 'I cannot fulfill this request directly. My function is to translate natural language into SQL queries that retrieve data. I cannot generate visualizations or graphical structures; that would be the role of a separate data visualization tool using the data retrieved by a SQL query.'}